# 01 · Clean raw MedDialog (COVID-EN) data and prepare it for patient-model fine-tuning

**SimClinic, CS425 Group 7.** This notebook turns `data/raw/english-{train,dev,test}.json` into:

1. `data/processed/clean/{split}.jsonl`: cleaned dialogues that do not depend on any model (for reuse by other pipelines).
2. `data/processed/tokenized/<model_slug>/`: a tokenized `DatasetDict` with `input_ids`, `attention_mask` and `labels`, where the loss is computed **only on the patient's turns**.

**To try another Hugging Face model, change `MODEL_NAME` in the config cell and re-run all cells.** Each model gets its own output folder.

Training string format, following the proposal:
```
<|doctor|>what brings you in today?<|patient|>i have had a dry cough for 3 days ...<|doctor|>
└────────── masked (context) ───────┘└marker┘└──── loss computed here ──────────┘└ loss ┘
```
The patient learns to end its turn by emitting the next role token (`<|doctor|>`). At inference you stop generating on that token.

## 0. Config
This is the only cell you normally need to edit.

In [1]:
from pathlib import Path

# --- model -------------------------------------------------------------------
MODEL_NAME = "gpt2"          # try: "gpt2-medium", "distilgpt2", "EleutherAI/pythia-160m", "Qwen/Qwen2.5-0.5B"
MAX_LENGTH = 512             # clipped to the tokenizer's model_max_length (GPT-2: 1024)

# --- training target -----------------------------------------------------------
TRAIN_ROLE = "patient"       # loss is computed only on this role's turns
ADD_DOCTOR_OPENER = True     # prepend a doctor question so the first patient turn is a reply, as in the proposal format
DOCTOR_OPENER = "what brings you in today?"
TRIM_AFTER_LAST_TARGET = True  # drop turns after the last TRAIN_ROLE turn (they would be masked anyway)

ROLE_TOKENS = {"doctor": "<|doctor|>", "patient": "<|patient|>"}
# Registered now so the vocabulary stays the same once the synthetic case-card data arrives.
EXTRA_SPECIAL_TOKENS = ["<|disease|>", "<|symptoms|>", "<|persona|>", "<|diagnosis|>"]

# --- cleaning / filtering --------------------------------------------------------
MIN_WORDS = 5                # first patient turn must have at least this many words
MAX_WORDS = 300              # drop dialogues with a patient turn longer than this
# Non-English filter: drop when patient text has more of these foreign function words than English ones.
# (MedDialog-COVID has a few Afrikaans and Spanish messages.) Extend for other languages.
FOREIGN_FUNCTION_WORDS = set("ek het die nie ons maar ook vir 'n sal wat jy hulle moontlik "
                             "el la que los las tengo una dos por con del es muy pero".split())

# Regexes removed from DOCTOR turns (applied in order, case-insensitive, after normalisation).
DOCTOR_BOILERPLATE = [
    r"would you like to video or text chat with me\??",
    r"^in brief:\s*",
    r"^thanks? (?:you )?for your (?:question|query|concern)[^.]*\.\s*",
    r"^hello and welcome to '?ask a doctor'? service\.\s*",
    r"^i have reviewed your query and here is my advice\.\s*",
    r"^hello dear\s*(?:[a-z]+\s*)?,?\s*(?:hi)?\s*(?:warm )?welcome to healthcaremagic\.com\s*",
    r"^respected user\s*,\s*hi\s*thanks for using [^.]*\.\s*",
    r"^i (?:can )?understand your concern\.\s*",
    r"hope i have answered your query\.?\s*",
    r"let me know if i can assist you further\.?\s*",
    r"(?:take care\s*)?regards(?!\s+to\b)[\s,.]*(?:dr\.?\s*[a-z .,]*)?$",   # sign-off + doctor name
]

# --- paths ---------------------------------------------------------------------
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
SPLITS = ["train", "dev", "test"]
SEED = 42

## 1. Load and validate the raw data

In [2]:
import json, re, unicodedata
from collections import Counter
import pandas as pd

pd.set_option("display.max_colwidth", 120)

raw = {s: json.loads((RAW_DIR / f"english-{s}.json").read_text(encoding="utf-8")) for s in SPLITS}

ROLE_RE = re.compile(r"^(patient|doctor):\s?(.*)$", re.S)
for split, items in raw.items():
    for i, item in enumerate(items):
        assert set(item) == {"description", "utterances"}, (split, i, item.keys())
        roles = [ROLE_RE.match(u).group(1) for u in item["utterances"]]  # fails loudly on a bad prefix
        assert roles[0] == "patient", (split, i)
        assert all(a != b for a, b in zip(roles, roles[1:])), f"roles do not alternate: {split}[{i}]"

pd.DataFrame({
    s: {"dialogues": len(d),
        "turns": sum(len(x["utterances"]) for x in d),
        "multi-turn dialogues": sum(len(x["utterances"]) > 2 for x in d)}
    for s, d in raw.items()
}).T

,dialogues,turns,multi-turn dialogues
train,482,981,2
dev,60,126,1
test,61,122,0


In [3]:
raw["train"][0]

{'description': 'throat a bit sore and want to get a good imune booster, especially in light of the virus. please advise. have not been in contact with nyone with the virus.',
 'utterances': ['patient: throat a bit sore and want to get a good imune booster, especially in light of the virus. please advise. have not been in contact with nyone with the virus.',
  "doctor: during this pandemic. throat pain can be from a strep throat infection (antibiotics needed), a cold or influenza or other virus, or from some other cause such as allergies or irritants. usually, a person sees the doctor (call first) if the sore throat is bothersome, recurrent, or doesn't go away quickly. covid-19 infections tend to have cough, whereas strep throat usually lacks cough but has more throat pain. (3/21/20)"]}

## 2. Parse turns
Split each `"role: text"` string into `{role, text}`, and give every dialogue a stable id.

In [4]:
def parse(split, idx, item):
    turns = [dict(zip(("role", "text"), ROLE_RE.match(u).groups())) for u in item["utterances"]]
    return {"dialogue_id": f"{split}-{idx:04d}", "split": split,
            "description": item["description"], "raw_turns": turns}

dialogues = {s: [parse(s, i, x) for i, x in enumerate(items)] for s, items in raw.items()}
dialogues["train"][2]["raw_turns"][0]

{'role': 'patient',
 'text': 'i have a tight and painful chest with a dry cough, no fever and no headaches. could it possibly be coronavirus?'}

## 3. Normalise text
These steps apply to both roles. The text is already lowercase. Typos and hedging are kept on purpose, because that informal style is what the patient model should learn.

In [5]:
CHAR_MAP = str.maketrans({"‘": "'", "’": "'", "“": '"', "”": '"', "`": "'",
                          "–": "-", "—": " - ", " ": " "})
URL_RE = re.compile(r"(?:https?://|www\.)\S+|\b[\w-]+\.(?:com|org|net|gov)(?:/\S*)?", re.I)
MISSING_SPACE_RE = re.compile(r"(?<=[a-z]{3}[.?!,])(?=[a-z])")  # "magic.i can" -> "magic. i can"

def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).translate(CHAR_MAP)
    text = "".join(ch for ch in text if unicodedata.category(ch)[0] != "C" or ch in "\n\t")
    text = URL_RE.sub(" ", text)
    text = MISSING_SPACE_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)      # "word ." -> "word."
    text = re.sub(r"([.,!?;:])\1{2,}", r"\1\1", text)  # "!!!!!" -> "!!"
    return text.strip(" -:")

print(clean_text("thanks for your question on healthcare magic.i can understand your concern . see https://www.healthtap.com/blog x"))

thanks for your question on healthcare magic. i can understand your concern. see x


## 4. Remove doctor boilerplate
Doctor turns are only context for the patient model, but removing stock greetings, sign-offs and doctor names keeps the context focused on content. Extend `DOCTOR_BOILERPLATE` in the config cell when you find new patterns.

In [6]:
BOILERPLATE_RES = [re.compile(p, re.I) for p in DOCTOR_BOILERPLATE]

def strip_boilerplate(text: str) -> str:
    for _ in range(3):                      # openers can stack ("thanks for your query. i understand ...")
        before = text
        for rx in BOILERPLATE_RES:
            text = rx.sub("", text).strip(" ,.-")
        if text == before:
            break
    return text.strip()

def clean_turn(turn):
    text = clean_text(turn["text"])
    if turn["role"] == "doctor":
        text = strip_boilerplate(text)
    return {"role": turn["role"], "text": text}

for split in SPLITS:
    for d in dialogues[split]:
        d["turns"] = [clean_turn(t) for t in d["raw_turns"]]

ex = next(d for d in dialogues["train"] if "healthcare magic" in d["raw_turns"][1]["text"])
print("RAW    :", ex["raw_turns"][1]["text"][:300], "\n")
print("CLEANED:", ex["turns"][1]["text"][:300])

RAW    : thanks for your question on healthcare magic.i can understand your concern. pneumonia with pregnancy is always critical.antibiotics as early as possible is the treatment of choice for pneumonia. so better to immediately consult doctor and start antibiotics for pneumonia. drink plenty of fluids orall 

CLEANED: pneumonia with pregnancy is always critical. antibiotics as early as possible is the treatment of choice for pneumonia. so better to immediately consult doctor and start antibiotics for pneumonia. drink plenty of fluids orally and keep yourself hydrated. with prompt and appropriate treatment, pneumo


## 5. Filter and deduplicate
Each dropped dialogue is logged with a reason. Deduplication uses a normalised key (punctuation and spacing removed). Train examples that also appear in dev or test are removed, so the evaluation is not contaminated.

In [7]:
EN_FUNCTION_WORDS = set("i me my the a an and or but is am are was were be been have has had do does did "
                        "to of in on for with at from it this that what how can should will would not no "
                        "you your he she his her they we".split())

def words(text): return text.split()
def norm_key(d):
    target = " ".join(t["text"] for t in d["turns"] if t["role"] == TRAIN_ROLE)
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", target.lower()).split())

def drop_reason(d):
    pt = [t["text"] for t in d["turns"] if t["role"] == TRAIN_ROLE]
    if not pt or any(not t for t in pt):                    return "empty patient turn"
    if any(not t["text"] for t in d["turns"]):              return "empty doctor turn after cleaning"
    if len(words(pt[0])) < MIN_WORDS:                       return f"first patient turn < {MIN_WORDS} words"
    if any(len(words(t)) > MAX_WORDS for t in pt):          return f"patient turn > {MAX_WORDS} words"
    toks = [w.strip(".,!?()") for w in words(" ".join(pt).lower())]
    if sum(w in FOREIGN_FUNCTION_WORDS for w in toks) > sum(w in EN_FUNCTION_WORDS for w in toks):
        return "likely non-English"
    return None

kept, drops = {}, []
for split in SPLITS:
    seen, kept[split] = set(), []
    for d in dialogues[split]:
        reason = drop_reason(d)
        key = norm_key(d)
        if reason is None and key in seen:
            reason = "duplicate within split"
        if reason:
            drops.append({"split": split, "dialogue_id": d["dialogue_id"], "reason": reason,
                          "patient_text": d["turns"][0]["text"][:100]})
            continue
        seen.add(key)
        kept[split].append(d)

held_out = {norm_key(d) for s in ("dev", "test") for d in kept[s]}
leaked = [d for d in kept["train"] if norm_key(d) in held_out]
for d in leaked:
    drops.append({"split": "train", "dialogue_id": d["dialogue_id"], "reason": "overlaps dev/test",
                  "patient_text": d["turns"][0]["text"][:100]})
kept["train"] = [d for d in kept["train"] if norm_key(d) not in held_out]

drops_df = pd.DataFrame(drops, columns=["split", "dialogue_id", "reason", "patient_text"])
drops_df.pivot_table(index="reason", columns="split", values="dialogue_id", aggfunc="count", fill_value=0)

split,dev,train
reason,,
duplicate within split,0,12
first patient turn < 5 words,0,2
likely non-English,1,2
patient turn > 300 words,2,2


In [8]:
drops_df.head(20)

,split,dialogue_id,reason,patient_text
0,train,train-0011,likely non-English,"ek het loop neus met slym hoes. moontlik allergie, hulle besig met verbouings by werk. ek het ook ho"
1,train,train-0136,duplicate within split,what is the treatment for covid-19?
2,train,train-0145,patient turn > 300 words,i have had a percocet addiction for 35 years and have had many dr.s in that time span. i was up to w
3,train,train-0151,patient turn > 300 words,"hello doctor, last night i was getting chills, feeling more cold than usual and lightheaded and a lo"
4,train,train-0157,first patient turn < 5 words,criteria for covid test?
5,train,train-0172,duplicate within split,do i have covid 19?
6,train,train-0173,likely non-English,ons het teruggekom vanaf oostenryk op 8 maart..reeds 13 dae gelede. het tans seer keel maar ook nasa
7,train,train-0215,duplicate within split,"hi have been having a mild fever since the 15 of march, with a strange sensation in the throat, feve"
8,train,train-0276,duplicate within split,can covid-19 be transmitted sexually? can any coronavirus be transmitted sexually?
9,train,train-0281,duplicate within split,anxiety because of the covid 19 news?


## 6. Before / after report

In [9]:
def length_stats(ds, key, role):
    lens = pd.Series([len(t["text"].split()) for d in ds for t in d[key] if t["role"] == role])
    return lens.describe(percentiles=[.5, .95]).round(1)[["count", "mean", "50%", "95%", "max"]]

report = pd.concat({
    (s, stage, role): length_stats(dialogues[s] if stage == "raw" else kept[s],
                                   "raw_turns" if stage == "raw" else "turns", role)
    for s in SPLITS for stage in ("raw", "clean") for role in ("patient", "doctor")
}, axis=1).T
report.index.names = ["split", "stage", "role"]
report

count  mean   50%    95%    max
split stage role                                    
train raw   patient  491.0  46.6  38.0  117.5  633.0
            doctor   490.0  57.4  57.0  101.0  252.0
      clean patient  465.0  43.7  38.0  107.4  256.0
            doctor   465.0  49.8  49.0   89.8  209.0
dev   raw   patient   63.0  60.4  45.0  183.5  401.0
            doctor    63.0  56.3  52.0   96.8  162.0
      clean patient   60.0  50.6  45.0  121.1  209.0
            doctor    60.0  47.0  45.0   81.2  155.0
test  raw   patient   61.0  43.8  41.0   88.0  201.0
            doctor    61.0  59.6  65.0  105.0  117.0
      clean patient   61.0  43.8  41.0   88.0  202.0
            doctor    61.0  52.5  58.0   96.0  110.0

In [10]:
for d in kept["train"][:2]:
    for raw_t, t in zip(d["raw_turns"], d["turns"]):
        print(f"[{t['role']}] RAW  : {raw_t['text'][:220]}")
        print(f"[{t['role']}] CLEAN: {t['text'][:220]}\n")
    print("-" * 100)

[patient] RAW  : throat a bit sore and want to get a good imune booster, especially in light of the virus. please advise. have not been in contact with nyone with the virus.
[patient] CLEAN: throat a bit sore and want to get a good imune booster, especially in light of the virus. please advise. have not been in contact with nyone with the virus.

[doctor] RAW  : during this pandemic. throat pain can be from a strep throat infection (antibiotics needed), a cold or influenza or other virus, or from some other cause such as allergies or irritants. usually, a person sees the doctor 
[doctor] CLEAN: during this pandemic. throat pain can be from a strep throat infection (antibiotics needed), a cold or influenza or other virus, or from some other cause such as allergies or irritants. usually, a person sees the doctor 

----------------------------------------------------------------------------------------------------
[patient] RAW  : hey there i have had cold "symptoms" for over a week and h

In [11]:
# Sanity checks: no leftover URLs or boilerplate in the cleaned text.
all_text = [t["text"] for s in SPLITS for d in kept[s] for t in d["turns"]]
assert not any(URL_RE.search(t) for t in all_text), "URL left in cleaned text"
leftover = [t for t in all_text if re.search(r"video or text chat|healthcare ?magic|^in brief:", t)]
assert not leftover, leftover[:3]
print("clean OK", {s: len(kept[s]) for s in SPLITS})

clean OK {'train': 464, 'dev': 57, 'test': 61}


## 7. Format training strings
Each dialogue becomes one string. The character spans of the target role's turns are recorded at the same time, so the loss mask comes from those spans and not from token-id heuristics.

In [12]:
def format_dialogue(turns, eos_token):
    """Return (text, spans): spans are [start, end) character ranges to compute loss on."""
    turns = list(turns)
    if ADD_DOCTOR_OPENER and turns[0]["role"] != "doctor":
        turns = [{"role": "doctor", "text": DOCTOR_OPENER}] + turns
    if TRIM_AFTER_LAST_TARGET:
        last = max(i for i, t in enumerate(turns) if t["role"] == TRAIN_ROLE)
        turns = turns[: last + 2]            # keep the next turn's marker as the stop signal

    text, spans = "", []
    for i, t in enumerate(turns):
        marker = ROLE_TOKENS[t["role"]]
        is_stop_only = TRIM_AFTER_LAST_TARGET and i == len(turns) - 1 and t["role"] != TRAIN_ROLE
        if is_stop_only:
            spans[-1][1] = len(text) + len(marker)   # the next role marker ends the target turn
            text += marker
            break
        text += marker
        start = len(text)
        text += t["text"]
        if t["role"] == TRAIN_ROLE:
            spans.append([start, len(text)])
        elif spans and spans[-1][1] == start - len(marker):
            spans[-1][1] = start                       # include this marker in the previous target span
    else:
        text += eos_token
        if turns[-1]["role"] == TRAIN_ROLE:
            spans[-1][1] = len(text)
    return text, spans

demo_text, demo_spans = format_dialogue(kept["train"][0]["turns"], "<|endoftext|>")
print(demo_text[:400], "\n")
print("trained spans:", [demo_text[a:b][:80] for a, b in demo_spans])

<|doctor|>what brings you in today?<|patient|>throat a bit sore and want to get a good imune booster, especially in light of the virus. please advise. have not been in contact with nyone with the virus.<|doctor|>

 

trained spans: ['throat a bit sore and want to get a good imune booster, especially in light of t']


## 8. Tokenizer (model-agnostic)

In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
assert tokenizer.is_fast, "a fast tokenizer is needed for offset mapping"
base_vocab = len(tokenizer)
n_added = tokenizer.add_special_tokens(
    {"additional_special_tokens": list(ROLE_TOKENS.values()) + EXTRA_SPECIAL_TOKENS})
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token          # GPT-2 has no pad token
tokenizer.padding_side = "right"
MAX_LEN = min(MAX_LENGTH, tokenizer.model_max_length)
MODEL_SLUG = MODEL_NAME.replace("/", "__")

print(f"{MODEL_NAME}: vocab {base_vocab} -> {len(tokenizer)} (+{n_added}); eos={tokenizer.eos_token!r} "
      f"pad={tokenizer.pad_token!r}; max_len={MAX_LEN}")
print("-> in training: model.resize_token_embeddings(len(tokenizer))")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

C:\Users\Marcus\Documents\GitHub\cs425\model\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Marcus\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

gpt2: vocab 50257 -> 50263 (+6); eos='<|endoftext|>' pad='<|endoftext|>'; max_len=512
-> in training: model.resize_token_embeddings(len(tokenizer))


## 9. Tokenize and build the loss mask
`labels` is a copy of `input_ids`, with `-100` on every token that does not start inside a target span. The `<|patient|>` marker itself is masked, because it is part of the prompt.

In [14]:
def encode(d):
    text, spans = format_dialogue(d["turns"], tokenizer.eos_token)
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN, return_offsets_mapping=True)
    labels = [tok if (e > s and any(a <= s < b for a, b in spans)) else -100
              for tok, (s, e) in zip(enc["input_ids"], enc["offset_mapping"])]
    full_len = len(tokenizer(text)["input_ids"])
    return {"dialogue_id": d["dialogue_id"], "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"], "labels": labels,
            "n_train_tokens": sum(l != -100 for l in labels), "truncated": full_len > MAX_LEN}

encoded = {s: [encode(d) for d in kept[s]] for s in SPLITS}
empty = {s: [e["dialogue_id"] for e in encoded[s] if e["n_train_tokens"] == 0] for s in SPLITS}
encoded = {s: [e for e in encoded[s] if e["n_train_tokens"] > 0] for s in SPLITS}

pd.DataFrame({s: {
    "examples": len(encoded[s]),
    "dropped (no trainable tokens)": len(empty[s]),
    "truncated": sum(e["truncated"] for e in encoded[s]),
    "tokens p50": int(pd.Series([len(e["input_ids"]) for e in encoded[s]]).median()),
    "tokens p95": int(pd.Series([len(e["input_ids"]) for e in encoded[s]]).quantile(.95)),
    "tokens max": max(len(e["input_ids"]) for e in encoded[s]),
    "trainable tokens": sum(e["n_train_tokens"] for e in encoded[s]),
} for s in SPLITS}).T

,examples,dropped (no trainable tokens),truncated,tokens p50,tokens p95,tokens max,trainable tokens
train,464,0,0,57,148,327,26028
dev,57,0,0,65,205,437,3900
test,61,0,0,58,125,240,3350


In [15]:
def show_mask(e, limit=120):
    """Trained tokens are shown in «», masked tokens plain."""
    out = []
    for tok, lab in list(zip(e["input_ids"], e["labels"]))[:limit]:
        piece = tokenizer.decode([tok])
        out.append(f"«{piece}»" if lab != -100 else piece)
    print("".join(out))

show_mask(encoded["train"][0])

<|doctor|>what brings you in today?<|patient|>«thro»«at»« a»« bit»« sore»« and»« want»« to»« get»« a»« good»« im»«une»« booster»«,»« especially»« in»« light»« of»« the»« virus»«.»« please»« advise»«.»« have»« not»« been»« in»« contact»« with»« n»«y»«one»« with»« the»« virus»«.»«<|doctor|>»


In [16]:
# Check: each contiguous run of trained tokens decodes to patient text (plus the stop marker / eos).
target_texts = {d["dialogue_id"]: " ".join(t["text"] for t in d["turns"] if t["role"] == TRAIN_ROLE)
                for s in SPLITS for d in kept[s]}
stop_strings = [ROLE_TOKENS[r] for r in ROLE_TOKENS if r != TRAIN_ROLE] + [tokenizer.eos_token]

def runs(e):
    run = []
    for tok, lab in zip(e["input_ids"], e["labels"]):
        if lab != -100: run.append(tok)
        elif run: yield run; run = []
    if run: yield run

for s in SPLITS:
    for e in encoded[s]:
        assert len(e["input_ids"]) <= MAX_LEN
        for r in runs(e):
            txt = tokenizer.decode(r)
            for st in stop_strings: txt = txt.replace(st, "")
            assert txt.strip() in target_texts[e["dialogue_id"]], (e["dialogue_id"], txt[:120])
print("mask OK")

mask OK


## 10. Save

In [17]:
from datasets import Dataset, DatasetDict

clean_dir = OUT_DIR / "clean"
clean_dir.mkdir(parents=True, exist_ok=True)
for s in SPLITS:
    with open(clean_dir / f"{s}.jsonl", "w", encoding="utf-8") as f:
        for d in kept[s]:
            f.write(json.dumps({k: d[k] for k in ("dialogue_id", "split", "description", "turns")},
                               ensure_ascii=False) + "\n")
drops_df.to_csv(clean_dir / "dropped.csv", index=False)

tok_dir = OUT_DIR / "tokenized" / MODEL_SLUG
cols = ["dialogue_id", "input_ids", "attention_mask", "labels"]
dsd = DatasetDict({s: Dataset.from_list([{k: e[k] for k in cols} for e in encoded[s]]) for s in SPLITS})
dsd.save_to_disk(str(tok_dir / "dataset"))
tokenizer.save_pretrained(str(tok_dir / "tokenizer"))

prep_config = {
    "model_name": MODEL_NAME, "max_length": MAX_LEN, "train_role": TRAIN_ROLE,
    "add_doctor_opener": ADD_DOCTOR_OPENER, "doctor_opener": DOCTOR_OPENER,
    "trim_after_last_target": TRIM_AFTER_LAST_TARGET, "role_tokens": ROLE_TOKENS,
    "extra_special_tokens": EXTRA_SPECIAL_TOKENS, "vocab_size": len(tokenizer),
    "stop_token": ROLE_TOKENS["doctor" if TRAIN_ROLE == "patient" else "patient"],
    "min_words": MIN_WORDS, "max_words": MAX_WORDS,
    "doctor_boilerplate": DOCTOR_BOILERPLATE, "seed": SEED,
    "counts": {s: len(encoded[s]) for s in SPLITS},
    "dropped": drops_df.groupby(["split", "reason"]).size().rename("n").reset_index().to_dict("records"),
}
(tok_dir / "prep_config.json").write_text(json.dumps(prep_config, indent=2), encoding="utf-8")
print("saved ->", clean_dir, "and", tok_dir)
dsd

Saving the dataset (0/1 shards):   0%|          | 0/464 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/57 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/61 [00:00<?, ? examples/s]

saved -> C:\Users\Marcus\Documents\GitHub\cs425\model\data\processed\clean and C:\Users\Marcus\Documents\GitHub\cs425\model\data\processed\tokenized\gpt2


DatasetDict({
    train: Dataset({
        features: ['dialogue_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 464
    })
    dev: Dataset({
        features: ['dialogue_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 57
    })
    test: Dataset({
        features: ['dialogue_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 61
    })
})

## 11. Next steps: using this in the training notebook
```python
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq

tok_dir = "data/processed/tokenized/gpt2"
ds = load_from_disk(f"{tok_dir}/dataset").remove_columns("dialogue_id")
tokenizer = AutoTokenizer.from_pretrained(f"{tok_dir}/tokenizer")
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))
collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)  # pads labels with -100
# Trainer(model, args, train_dataset=ds["train"], eval_dataset=ds["dev"], data_collator=collator)
```
At generation time, stop on `prep_config["stop_token"]` (`<|doctor|>`).

**Later data.** The synthetic case-card dialogues from the proposal can reuse steps 7–10 unchanged. Prefix each string with the hidden header (`<|disease|>flu<|symptoms|>fever; cough<|persona|>34, f, chinese, teacher, anxious`), which stays masked, and split 80/10/10 by case card, stratified by disease.